# R06 Contrarian Round - probe evaluation and the H28 scramble test

**Author**: Knowledge Graph Foundry autonomous build

Evaluates the R05 campaign graph (benchmark article corpus, local gpt-oss-120b extractor, neo4j3) against the 47-probe set, then runs the R06-H28 falsification: permute all entity type labels by a fixed derangement, re-run the probes, restore, and measure the QA delta. A delta below 5% is evidence that typing is causally inert for retrieval and earns its keep only as the H23 audit baseline; a delta at or above 5% is the first causal evidence in the program that curing pays at retrieval time. The scramble is reversible - the label mapping is kept and restored, so the campaign graph is unchanged after the test.

In [ ]:
# imports
import datetime
import json
import os
import random
import re
from collections import Counter
from pathlib import Path

import yaml

# campaign graph lives on neo4j3 - override before Foundry reads settings
os.environ["NEO4J_URI"] = "bolt://user-konrad.jelen-kgf-neo4j3:7687"
os.environ["NEO4J_USER"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "kgfoundry"

from knowledge_graph_foundry import Foundry, load_settings

CONFIG = Path("../config-apnea.yml")
PROBES = Path("../tests/probes/apnea-probe-set.yml")

In [ ]:
# corrected content-word scorer for prose gold answers
# CPAP probes carry short value tokens; the apnea probes carry multi-sentence prose,
# so value-token matching does not apply. Score by content-word overlap (recall of the
# gold answer's distinctive content words in the produced answer), with a stopword filter.

STOP = set(
    "a an the and or but if then than that this these those of to in on at for with "
    "as by from into over under is are was were be been being it its they them their "
    "which who whom whose what when where why how may might can could would should will "
    "not no do does did has have had you your our we he she his her not more most some "
    "such about above below between during due while also both each other any all more".split()
)


def content_words(text):
    toks = re.findall(r"[a-z][a-z0-9\-]{2,}", text.casefold())
    return [t for t in toks if t not in STOP]


def content_overlap(gold_answer, produced):
    gold = set(content_words(gold_answer))
    if not gold:
        return 0.0
    prod = set(content_words(produced))
    return len(gold & prod) / len(gold)


REFUSAL_MARKERS = (
    "not in the graph",
    "does not contain",
    "no information",
    "cannot answer",
    "not available",
    "not found",
    "insufficient",
    "unable to",
    "i don't have",
    "not present",
    "no relevant",
)


def is_refusal(answer):
    a = answer.casefold()
    return any(m in a for m in REFUSAL_MARKERS)


# sanity: a gold answer scored against itself is 1.0; against an empty string is 0.0
_p = yaml.safe_load(PROBES.read_text())["probes"][0]
assert content_overlap(_p["gold_answer"], _p["gold_answer"]) == 1.0
assert content_overlap(_p["gold_answer"], "") == 0.0
print("scorer sanity ok; content-words in AQ01 gold:", len(set(content_words(_p["gold_answer"]))))

In [ ]:
# split probes by cluster membership in the ingested corpus
# a probe is in-wave (answerable) once its source cluster has been ingested; otherwise
# it is an out-of-wave control that must be refused. Ingested clusters are read live
# from KGFDocument.name (filenames encode the cluster as the c<N>_ prefix), so the
# split tracks reality wave by wave.

probes = yaml.safe_load(PROBES.read_text())["probes"]
CORRECT_THRESHOLD = 0.40  # content-overlap at/above this counts as answered

_CLUSTER_RE = re.compile(r"^c(\d+)_")


def ingested_clusters():
    with Foundry(load_settings(CONFIG)) as f:
        with f._driver.session() as s:
            names = [r["n"] for r in s.run(
                "MATCH (dd:KGFDocument) RETURN dd.name AS n"
            ).data()]
    out = set()
    for n in names:
        m = _CLUSTER_RE.match(n or "")
        if m:
            out.add(int(m.group(1)))
    return out


def split_probes():
    present = ingested_clusters()
    in_wave = [p for p in probes if p["cluster"] in present]
    out_wave = [p for p in probes if p["cluster"] not in present]
    return in_wave, out_wave, present


# preview the split (cheap graph read)
try:
    _in, _out, _present = split_probes()
    print("ingested clusters:", sorted(_present))
    print(f"in-wave (answerable) probes: {len(_in)}  ->", [p['id'] for p in _in])
    print(f"out-of-wave (refuse) controls: {len(_out)}")
except Exception as exc:
    print("graph not reachable yet:", exc)

In [ ]:
# probe runner - one open Foundry, query each probe, score prose overlap + refusal
def run_probes(probe_list, label):
    rows = []
    with Foundry(load_settings(CONFIG)) as f:
        for p in probe_list:
            ans = f.query(p["question"])["answer"]
            ov = content_overlap(p["gold_answer"], ans)
            rows.append({
                "id": p["id"],
                "cluster": p["cluster"],
                "overlap": ov,
                "correct": ov >= CORRECT_THRESHOLD,
                "refused": is_refusal(ans),
                "answer": ans,
            })
            print(f"{p['id']} c{p['cluster']:<3} overlap={ov:.2f} "
                  f"{'OK' if rows[-1]['correct'] else 'refuse' if rows[-1]['refused'] else 'miss'}")
    n = len(rows) or 1
    return {
        "label": label,
        "rows": rows,
        "accuracy": sum(r["correct"] for r in rows) / n,
        "refusal_rate": sum(r["refused"] for r in rows) / n,
        "mean_overlap": sum(r["overlap"] for r in rows) / n,
    }

## R05-H26 baseline - answerability tracks cluster membership

Run the full 47-probe set against the current campaign graph. In-wave probes (source cluster ingested) should answer; out-of-wave controls should refuse. This is the pre-scramble baseline that H28

In [ ]:
# run the baseline once in-wave probes exist (wave 3+). Skips cleanly on a probe-desert wave.
in_wave, out_wave, present = split_probes()
print(f"clusters ingested: {sorted(present)}  in-wave probes: {len(in_wave)}")

baseline = None
out_control = run_probes(out_wave, "baseline_out_of_wave")
print(f"\nout-of-wave refusal rate: {out_control['refusal_rate']:.2%} "
      f"(bar >= 70%); false-answer rate {1 - out_control['refusal_rate']:.2%}")

if in_wave:
    baseline = run_probes(in_wave, "baseline_in_wave")
    print(f"\nin-wave accuracy: {baseline['accuracy']:.2%}  "
          f"mean overlap {baseline['mean_overlap']:.3f}  (H25 bar >= 0.75)")
else:
    print("\nno in-wave probes yet - H28 scramble deferred until a probe-bearing cluster lands (wave 3+)")

## R06-H28 - the scramble test\n\nType labels are live Neo4j labels rendered into the reader's context (`## name (Type1, Type2)` in `_retrieve_local`), so permuting them is a clean isolation of typing's causal effect: same nodes, edges, embeddings, descriptions and properties, only the type strings changed. The scramble snapshots each entity's original labels to a temporary `_orig_types` property, applies a fixed derangement (no label maps to itself), and restores from the snapshot afterwards - the campaign graph is byte-identical before and after. A post-scramble in-wave accuracy within 5% of baseline falsifies the claim that ontology quality drives retrieval; a drop of 5% or more is the program's first causal evidence that curing pays at read time.

In [ ]:
# scramble machinery - reversible via per-node snapshot
STRUCTURAL = {"Entity", "Chunk", "KGFControl", "KGFLock", "KGFDocument",
              "KGFEntityVersion", "Query"}


def ontology_labels(driver):
    with driver.session() as s:
        alll = [r["label"] for r in s.run("CALL db.labels() YIELD label RETURN label")]
    return sorted(l for l in alll if l not in STRUCTURAL)


def derangement(labels, seed=1728):
    # a permutation with no fixed point; deterministic under the seed
    rng = random.Random(seed)
    assert len(labels) >= 2, "need >= 2 type labels to derange"
    while True:
        perm = labels[:]
        rng.shuffle(perm)
        if all(a != b for a, b in zip(labels, perm)):
            return dict(zip(labels, perm))


def scramble_types(driver, mapping):
    # snapshot ontology labels per node, relabel each node to its mapped set
    with driver.session() as s:
        s.run(
            "MATCH (e:Entity) "
            "WITH e, [l IN labels(e) WHERE NOT l IN $struct] AS ont "
            "SET e._orig_types = ont",
            struct=list(STRUCTURAL),
        )
        s.run(
            "MATCH (e:Entity) WHERE size(e._orig_types) > 0 "
            "WITH e, [l IN e._orig_types | $map[l]] AS newl "
            "CALL apoc.create.addLabels(e, newl) YIELD node AS n1 "
            "WITH e, newl "
            "WITH e, [l IN e._orig_types WHERE NOT l IN newl] AS toremove "
            "CALL apoc.create.removeLabels(e, toremove) YIELD node AS n2 "
            "RETURN count(*)",
            map=mapping,
        )


def restore_types(driver):
    # put the snapshot back, drop scrambled-in labels, clear the marker
    with driver.session() as s:
        s.run(
            "MATCH (e:Entity) WHERE e._orig_types IS NOT NULL "
            "CALL apoc.create.addLabels(e, e._orig_types) YIELD node "
            "RETURN count(*)"
        )
        s.run(
            "MATCH (e:Entity) WHERE e._orig_types IS NOT NULL "
            "WITH e, [l IN labels(e) WHERE NOT l IN $struct AND NOT l IN e._orig_types] AS extra "
            "WHERE size(extra) > 0 "
            "CALL apoc.create.removeLabels(e, extra) YIELD node "
            "RETURN count(*)",
            struct=list(STRUCTURAL),
        )
        s.run("MATCH (e:Entity) REMOVE e._orig_types")


def type_histogram(driver):
    with driver.session() as s:
        rows = s.run(
            "MATCH (e:Entity) UNWIND [l IN labels(e) WHERE NOT l IN $struct] AS l "
            "RETURN l, count(*) AS c ORDER BY c DESC",
            struct=list(STRUCTURAL),
        ).data()
    return {r["l"]: r["c"] for r in rows}


print("scramble helpers defined")

In [ ]:
# H28 execution: scramble -> probe -> restore -> verify -> delta
# runs only when in-wave probes exist (wave 3+); guarded so a probe-desert graph skips
scramble_result = None
if baseline is None:
    print("H28 skipped - no in-wave probes on the current graph yet")
else:
    with Foundry(load_settings(CONFIG)) as f:
        labels = ontology_labels(f._driver)
        before_hist = type_histogram(f._driver)
        mapping = derangement(labels)
        print("derangement:", mapping)
        scramble_types(f._driver, mapping)
        after_hist = type_histogram(f._driver)
        print("scrambled; label census moved:",
              sum(before_hist.get(k, 0) != after_hist.get(k, 0) for k in set(before_hist) | set(after_hist)),
              "of", len(set(before_hist) | set(after_hist)), "labels")

    try:
        scrambled = run_probes(in_wave, "h28_scrambled_in_wave")
    finally:
        # restore no matter what - the campaign graph must come back intact
        with Foundry(load_settings(CONFIG)) as f:
            restore_types(f._driver)
            restored_hist = type_histogram(f._driver)
        assert restored_hist == before_hist, "RESTORE FAILED - histograms differ"
        print("graph restored - label histogram identical to pre-scramble")

    delta = baseline["accuracy"] - scrambled["accuracy"]
    scramble_result = {
        "baseline_accuracy": baseline["accuracy"],
        "scrambled_accuracy": scrambled["accuracy"],
        "delta": delta,
        "mapping": mapping,
    }
    print(f"\nH28: baseline {baseline['accuracy']:.2%} -> scrambled {scrambled['accuracy']:.2%} "
          f"(delta {delta:+.2%})")
    print("verdict:", "typing causally inert for retrieval (< 5%) - audit-only value"
          if abs(delta) < 0.05 else "typing pays at retrieval time (>= 5%) - first causal evidence")

In [ ]:
# persist results for the experiments log
results = {
    "out_of_wave_control": {k: v for k, v in out_control.items() if k != "rows"},
    "in_wave_baseline": ({k: v for k, v in baseline.items() if k != "rows"} if baseline else None),
    "h28_scramble": scramble_result,
    "detail": {
        "out_of_wave": out_control["rows"],
        "in_wave": baseline["rows"] if baseline else [],
    },
    "clusters_ingested": sorted(present),
}
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
out = Path("../reports") / f"probe-eval-r06-{stamp}.json"
out.write_text(json.dumps(results, indent=2, default=str))
print("saved", out)